In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set a random seed for reproducibility
RANDOM_STATE = 42

In [ ]:
# Load telemetry data
# Using a fraction (e.g., 10%) to build the pipeline quickly
print("Loading and subsampling telemetry data...")
df_telemetry = pd.read_csv('data/data.csv').sample(frac=0.1, random_state=RANDOM_STATE)

In [7]:
# Parse the Timedate column to datetime objects
df_telemetry['timedate'] = pd.to_datetime(df_telemetry['timedate'])

# Extract year and month for our aggregation target later
df_telemetry['year'] = df_telemetry['timedate'].dt.year
df_telemetry['month'] = df_telemetry['timedate'].dt.month

In [24]:
print("Loading metadata and merging...")
df_devices = pd.read_csv('data/devices.csv')

# Merge on deviceld
df = df_telemetry.merge(df_devices, on='deviceId', how='left')

print(f"Data shape after merge: {df.shape}")

Loading metadata and merging...
Data shape after merge: (6449732, 24)


In [25]:
print("Engineering features...")

# 1. Temperature Differentials
# Internal vs External ambient temperature
df['delta_t2_t1'] = df['t2'] - df['t1']

# Source HEX temp difference (t4 - t3) [cite: 21]
df['delta_source_hex'] = df['t4'] - df['t3']

# Load HEX temp difference (t6 - t5) [cite: 21]
df['delta_load_hex'] = df['t6'] - df['t5']

# 2. Temporal Cyclicality (Sine/Cosine for hour and month)
df['hour'] = df['timedate'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

# Drop original datetime as models can't ingest it directly
df = df.drop(columns=['timedate', 'hour', "period"])

Engineering features...


In [26]:
print("Splitting available training data into Local Train and Local Validation...")

# Define our feature columns (excluding identifiers, targets, and raw time units)
features = [col for col in df.columns if col not in ['deviceId', 'x2', 'year', 'month']]

# Local Train: Oct 2024 - Feb 2025
local_train_mask = (df['year'] == 2024) | ((df['year'] == 2025) & (df['month'] <= 2))
# Local Validation: March 2025 - April 2025
local_val_mask = (df['year'] == 2025) & (df['month'].isin([3, 4]))

X_local_train = df[local_train_mask][features]
y_local_train = df[local_train_mask]['x2']

X_local_val = df[local_val_mask][features]
y_local_val = df[local_val_mask]['x2']

# Keep track of device and month for validation aggregation later
local_val_meta = df[local_val_mask][['deviceId', 'year', 'month', 'x2']].copy()

print(f"Local Train size: {X_local_train.shape[0]}")
print(f"Local Val size: {X_local_val.shape[0]}")

Splitting available training data into Local Train and Local Validation...
Local Train size: 2463633
Local Val size: 1002363


In [28]:
from lightgbm import LGBMRegressor, early_stopping

print("Training Local LightGBM model...")

local_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

local_model.fit(
    X_local_train, y_local_train,
    eval_set=[(X_local_val, y_local_val)],
    callbacks=[early_stopping(stopping_rounds=50)]
)

Training Local LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024347 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 670
[LightGBM] [Info] Number of data points in the train set: 2463633, number of used features: 24
[LightGBM] [Info] Start training from score 0.178073
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 0.00235192


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [30]:
from sklearn.metrics import mean_absolute_error

print("Predicting and aggregating on Local Validation...")

# Generate 5-minute predictions
local_val_meta['pred_x2'] = local_model.predict(X_local_val)

# Group by device, year, and month to calculate the monthly average target
monthly_preds = local_val_meta.groupby(['deviceId', 'year', 'month']).agg(
    actual_x2=('x2', 'mean'),
    predicted_x2=('pred_x2', 'mean')
).reset_index()

# Calculate the official competition metric (MAE)
local_mae = mean_absolute_error(monthly_preds['actual_x2'], monthly_preds['predicted_x2'])
print(f"Local Validation MAE (Monthly Average): {local_mae:.5f}")

Predicting and aggregating on Local Validation...
Local Validation MAE (Monthly Average): 0.00760


In [ ]:
print("Retraining on ALL available training data...")

# X_full and y_full contain all data from Oct 2024 to Apr 2025
X_full = df[features]
y_full = df['x2']

final_model = LGBMRegressor(
    n_estimators=local_model.best_iteration_, # Use the best iteration found during local validation
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_full, y_full)

print("Predicting on official Validation/Test periods (May-Oct 2025)...")

# Assuming `df_submission_features` contains the May-Oct 2025 data with your engineered features
df_submission_features['pred_x2'] = final_model.predict(df_submission_features[features])

# Aggregate predictions to monthly averages
final_submission = df_submission_features.groupby(['deviceld', 'year', 'month']).agg(
    prediction=('pred_x2', 'mean')
).reset_index()

# Format strictly to match: deviceld, year, month, prediction
final_submission = final_submission[['deviceld', 'year', 'month', 'prediction']]

# Save to CSV
final_submission.to_csv('submission.csv', index=False)
print("Final submission saved to submission.csv! Ready to upload.")